# Netflix Executive Dashboard — Streamlit-Ready Visualizations

**Dataset:** Netflix Movies & TV Shows (Kaggle, 8,807 titles)

This notebook builds interactive Plotly visualizations designed for deployment in a Streamlit dashboard. All charts export as standalone HTML files.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter

# Load and prep data
df = pd.read_csv('../data/netflix_titles.csv')
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['years_to_netflix'] = df['year_added'] - df['release_year']

# Parse multi-value fields
countries = []
for entry in df['country'].dropna():
    countries.extend([c.strip() for c in entry.split(',')])
country_counts = pd.Series(Counter(countries)).sort_values(ascending=False)

genres = []
for entry in df['listed_in'].dropna():
    genres.extend([g.strip() for g in entry.split(',')])
genre_counts = pd.Series(Counter(genres)).sort_values(ascending=False)

total = len(df)
movies = len(df[df['type'] == 'Movie'])
tv_shows = len(df[df['type'] == 'TV Show'])
movie_pct = movies / total * 100

lifecycle = df[(df['years_to_netflix'] >= 0) & (df['years_to_netflix'] <= 50)]
avg_lifecycle = lifecycle['years_to_netflix'].mean()
addition_by_year = df.groupby('year_added').size()
peak_year = int(addition_by_year.idxmax())
peak_count = int(addition_by_year.max())

print(f"KPI SUMMARY:")
print(f"  Total Titles: {total:,}")
print(f"  Movies: {movies:,} ({movie_pct:.1f}%)")
print(f"  TV Shows: {tv_shows:,} ({100-movie_pct:.1f}%)")
print(f"  Top Country: {country_counts.index[0]} ({country_counts.iloc[0]:,})")
print(f"  Top Genre: {genre_counts.index[0]} ({genre_counts.iloc[0]:,})")
print(f"  Avg Release→Netflix: {avg_lifecycle:.1f} years")
print(f"  Peak Addition Year: {peak_year} ({peak_count:,} titles)")

KPI SUMMARY:
  Total Titles: 8,807
  Movies: 6,131 (69.6%)
  TV Shows: 2,676 (30.4%)
  Top Country: United States (3,690)
  Top Genre: International Movies (2,752)
  Avg Release→Netflix: 4.4 years
  Peak Addition Year: 2019 (1,999 titles)


## 1. Content Portfolio Overview

Combined view: content mix pie chart, top countries bar chart, and addition timeline.

In [2]:
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}],
           [{"type": "bar", "colspan": 2}, None]],
    subplot_titles=('Content Mix', 'Top 10 Countries', 'Content Addition Timeline')
)

fig.add_trace(
    go.Pie(labels=['Movies', 'TV Shows'], values=[movies, tv_shows],
           marker_colors=['#E50914', '#221F1F'], hole=0.4),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=country_counts.head(10).values, y=country_counts.head(10).index,
           orientation='h', marker_color='#E50914'),
    row=1, col=2
)

addition_by_type = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)
fig.add_trace(
    go.Scatter(x=addition_by_type.index, y=addition_by_type['Movie'],
               mode='lines', name='Movies', line=dict(color='#E50914', width=2)),
    row=2, col=1
)
fig.add_trace(
    go.Scatter(x=addition_by_type.index, y=addition_by_type['TV Show'],
               mode='lines', name='TV Shows', line=dict(color='#221F1F', width=2)),
    row=2, col=1
)

fig.update_layout(height=700, showlegend=True, template='plotly_white',
                  title_text="Netflix Content Portfolio Overview")
fig.write_html('../output/01_portfolio_overview.html')
print("Saved: output/01_portfolio_overview.html")

Saved: output/01_portfolio_overview.html


## 2. Regional Content Heatmap

Country × Content Type matrix for the top 25 countries.

In [3]:
top25_countries = country_counts.head(25)
country_type = {}
for _, row in df.iterrows():
    if pd.isna(row['country']):
        continue
    for c in [c.strip() for c in row['country'].split(',')]:
        if c in top25_countries.index:
            if c not in country_type:
                country_type[c] = {'Movie': 0, 'TV Show': 0}
            country_type[c][row['type']] += 1

heatmap_df = pd.DataFrame(country_type).T.fillna(0).astype(int)
heatmap_df = heatmap_df.reindex(top25_countries.index)

fig = px.imshow(heatmap_df.T,
                labels=dict(x="Country", y="Content Type", color="Titles"),
                x=heatmap_df.index,
                y=heatmap_df.columns,
                color_continuous_scale='Reds',
                title='Regional Content Heatmap: Movies vs TV Shows')
fig.write_html('../output/02_regional_heatmap.html')
print("Saved: output/02_regional_heatmap.html")

Saved: output/02_regional_heatmap.html


## 3. Genre Opportunity Matrix

Scatter plot: genre saturation (titles) vs geographic spread (countries). Bubble size = TV ratio. Color = opportunity score.

In [4]:
genre_stats = {}
for _, row in df.iterrows():
    if pd.isna(row['listed_in']) or pd.isna(row['country']):
        continue
    for g in [g.strip() for g in row['listed_in'].split(',')]:
        if g not in genre_stats:
            genre_stats[g] = {'count': 0, 'countries': set(), 'tv_ratio': 0, 'movies': 0, 'tv': 0}
        genre_stats[g]['count'] += 1
        for c in [c.strip() for c in row['country'].split(',')]:
            genre_stats[g]['countries'].add(c)
        if row['type'] == 'Movie':
            genre_stats[g]['movies'] += 1
        else:
            genre_stats[g]['tv'] += 1

genre_matrix = pd.DataFrame({
    k: {
        'titles': v['count'],
        'countries': len(v['countries']),
        'tv_ratio': v['tv'] / v['count'] * 100,
        'opportunity': len(v['countries']) / v['count'] * 100
    }
    for k, v in genre_stats.items()
}).T

genre_matrix = genre_matrix.sort_values('titles', ascending=False).head(30)

fig = px.scatter(genre_matrix.reset_index(), x='titles', y='countries',
                 size='tv_ratio', color='opportunity',
                 hover_name='index',
                 labels={'titles': 'Total Titles', 'countries': 'Number of Countries',
                        'tv_ratio': 'TV Show %', 'opportunity': 'Opportunity Score'},
                 title='Genre Opportunity Matrix: Saturation vs Geographic Spread',
                 color_continuous_scale='RdYlGn',
                 size_max=30)
fig.write_html('../output/03_genre_opportunity.html')
print("Saved: output/03_genre_opportunity.html")

Saved: output/03_genre_opportunity.html


## 4. Acquisition Timeline

In [5]:
acq_df = df[df['years_to_netflix'].between(0, 20)].copy()
acq_by_year = acq_df.groupby(['release_year', 'type']).size().unstack(fill_value=0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=acq_by_year.index, y=acq_by_year['Movie'],
                         mode='lines', name='Movies', line=dict(color='#E50914', width=2)))
fig.add_trace(go.Scatter(x=acq_by_year.index, y=acq_by_year['TV Show'],
                         mode='lines', name='TV Shows', line=dict(color='#221F1F', width=2)))
fig.update_layout(
    title='Netflix Acquisition Timeline: Release Year to Platform',
    xaxis_title='Release Year',
    yaxis_title='Titles Acquired',
    template='plotly_white',
    xaxis_range=[1990, 2022]
)
fig.write_html('../output/04_acquisition_timeline.html')
print("Saved: output/04_acquisition_timeline.html")

Saved: output/04_acquisition_timeline.html


## 5. Regional Gap Analysis

In [6]:
global_avg = country_counts.mean()
gap_data = []
for country, count in country_counts.items():
    if count < global_avg and count >= 5:
        gap_data.append({
            'country': country,
            'titles': count,
            'gap': global_avg - count,
            'opportunity': (global_avg - count) / global_avg * 100
        })

gap_df = pd.DataFrame(gap_data).sort_values('opportunity', ascending=False).head(20)

fig = px.bar(gap_df, x='opportunity', y='country', orientation='h',
             color='titles',
             labels={'opportunity': 'Opportunity Score (% below global avg)',
                    'country': 'Country', 'titles': 'Current Titles'},
             title='Regional Content Gap: Markets Below Global Average',
             color_continuous_scale='Reds')
fig.write_html('../output/05_regional_gaps.html')
print("Saved: output/05_regional_gaps.html")

Saved: output/05_regional_gaps.html


---

All 5 interactive dashboard components have been exported as standalone HTML files in `../output/`.

Load any HTML file in a browser to view the interactive Plotly charts. These same visualizations are embedded in `dashboard.py` for Streamlit deployment.